In [1]:
from dotenv import load_dotenv

from Ch02_RAG.ch02_04_text_splitter import text_splitter

load_dotenv("../.env")

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

/Users/choyoungrae/Desktop/book-study/books/RAG 마스터 랭체인으로 완성하는 LLM 서비스/rag-master-practice-260326/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
url = "https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/refs/heads/main/Ch02.%20RAG/Data/2024_KB_%EB%B6%80%EB%8F%99%EC%82%B0_%EB%B3%B4%EA%B3%A0%EC%84%9C_%EC%B5%9C%EC%A2%85.pdf"
loader = PyPDFLoader(url)
pages = loader.load()

print("청크의 수:", len(pages))

청크의 수: 84


In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(pages)

print("분할된 청크의 수:", len(splits))

분할된 청크의 수: 135


In [5]:
chunk_lengths = [len(chunk.page_content) for chunk in splits]
max_length = max(chunk_lengths)
min_length = min(chunk_lengths)
avg_length = sum(chunk_lengths) / len(chunk_lengths)

print('청크의 최대 길이 :', max_length)
print('청크의 최소 길이 :', min_length)
print('청크의 평균 길이 :', avg_length)

청크의 최대 길이 : 1000
청크의 최소 길이 : 56
청크의 평균 길이 : 674.9481481481481


In [6]:
embedding_function = OpenAIEmbeddings()

vectordb = Chroma.from_documents(documents=splits, embedding=embedding_function)

In [7]:
print("문서의 수:", vectordb._collection.count())

문서의 수: 135


In [8]:
question = "수도권 주택 매매 전망"
top_threes_docs = vectordb.similarity_search(question, k=2)

for i, doc in enumerate(top_threes_docs, 1):
    print(f"문서 {i}:")
    print(f"내용: {doc.page_content[:150]}...")
    print(f"메타데이터: {doc.metadata}")
    print('--' * 20)

문서 1:
내용: 8 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
실 등에 따른 주택 경기 불안을 이유로 매매를 망설이며 시세 대비 저렴한 매물에만 관심을 보였다. 결
국 매도자와 매수자 간 희망가격 차이로 인한 매매 거래 위축 현상은 2023년 거래 침체의 가...
메타데이터: {'title': 'Morning Meeting', 'moddate': '2024-03-04T15:30:01+09:00', 'total_pages': 84, 'source': 'https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/refs/heads/main/Ch02.%20RAG/Data/2024_KB_%EB%B6%80%EB%8F%99%EC%82%B0_%EB%B3%B4%EA%B3%A0%EC%84%9C_%EC%B5%9C%EC%A2%85.pdf', 'page': 14, 'creationdate': '2024-03-04T15:30:01+09:00', 'author': '손은경', 'creator': 'Microsoft® Word 2016', 'page_label': '15', 'producer': 'Microsoft® Word 2016'}
----------------------------------------
문서 2:
내용: 18 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
그림Ⅰ-30. 수도권 입주물량과 전세가격 변동률 추이  그림Ⅰ-31. 기타지방 입주물량과 전세가격 변동률 추이 
 
 
 
자료: KB국민은행, 부동산114  자료: KB국민은행, 부동산114...
메타데이터: {'total_pages': 84, 'source': 'https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/refs/heads/main/Ch02.%20RAG/Data/2024_KB_%EB%B6%80%EB%8F%99%EC%82%B0_

In [9]:
question = "수도권 주택 매매 전망"
top_three_docs = vectordb.similarity_search_with_relevance_scores(question, k=3)

for i, doc in enumerate(top_three_docs, 1):
    print(f"문서 {i}:")
    print(f"유사 점수 {doc[1]}:")
    print(f"내용: {doc[0].page_content[:150]}...")
    print(f"메타데이터: {doc[0].metadata}")
    print('--' * 20)

문서 1:
유사 점수 0.8451337337601001:
내용: 8 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
실 등에 따른 주택 경기 불안을 이유로 매매를 망설이며 시세 대비 저렴한 매물에만 관심을 보였다. 결
국 매도자와 매수자 간 희망가격 차이로 인한 매매 거래 위축 현상은 2023년 거래 침체의 가...
메타데이터: {'creationdate': '2024-03-04T15:30:01+09:00', 'page_label': '15', 'creator': 'Microsoft® Word 2016', 'source': 'https://raw.githubusercontent.com/langchain-kr/langchain-tutorial/refs/heads/main/Ch02.%20RAG/Data/2024_KB_%EB%B6%80%EB%8F%99%EC%82%B0_%EB%B3%B4%EA%B3%A0%EC%84%9C_%EC%B5%9C%EC%A2%85.pdf', 'page': 14, 'producer': 'Microsoft® Word 2016', 'moddate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'total_pages': 84}
----------------------------------------
문서 2:
유사 점수 0.8328907591539737:
내용: 18 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
그림Ⅰ-30. 수도권 입주물량과 전세가격 변동률 추이  그림Ⅰ-31. 기타지방 입주물량과 전세가격 변동률 추이 
 
 
 
자료: KB국민은행, 부동산114  자료: KB국민은행, 부동산114...
메타데이터: {'creationdate': '2024-03-04T15:30:01+09:00', 'page': 24, 'source': 'https://raw.githubusercontent.com/langchain-k